# External validation on LungHist700
Tests the three models trained on LC25000 (from **unified_results.zip**) on an **independent** dataset,
LungHist700 (Diosdado et al., 2024; 691 images, 45 patients, 20x and 40x, CC BY 4.0). **No retraining.**

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Takes about 15–25 minutes.
If your earlier Colab session has closed, the notebook will ask you to upload **unified_results.zip**.
At the end it prints a **SUMMARY** and downloads **external_results.zip**. Send both to Claude.

**Label mapping:** LungHist700 adenocarcinoma (all differentiation grades) → lung_aca;
squamous cell carcinoma (all grades) → lung_scc; normal → lung_n.

**Pre-specified primary analysis:** tiled inference (12 tiles of 400 × 400 px per image, probabilities averaged),
no colour normalisation. Whole-image inference and Reinhard colour normalisation are sensitivity analyses.

### Settings

In [ ]:
import os
TEST_MODE = os.environ.get("EXT_TEST_MODE") == "1"     # used only for an offline test; leave as is
SEED = 42
IMG_SIZE = 160
TILE = 400                     # tile side in pixels on the 1200 x 1600 LungHist700 images
N_BOOT = 2000
REINHARD_REF_PER_CLASS = 100   # LC25000 images per class used to compute the colour reference
OUT_DIR = "/content/external_results"
RESULTS_DIR = "/content/unified_results"    # trained models from the unified run
DATA_ROOT = "/content/lunghist700"
LH_URL = "https://ndownloader.figshare.com/files/45206104"   # LungHist700.rar (670 MB, figshare)
if TEST_MODE:
    OUT_DIR, RESULTS_DIR, DATA_ROOT = os.environ["EXT_OUT"], os.environ["EXT_RESULTS"], os.environ["EXT_DATA"]
    REINHARD_REF_PER_CLASS, N_BOOT = 5, 200

### Imports

In [ ]:
import json, glob, re, shutil, subprocess, zipfile, random
import numpy as np, pandas as pd, cv2
import matplotlib
if TEST_MODE: matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf, keras
from keras import layers
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from statsmodels.stats.contingency_tables import mcnemar
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

### Get the trained models (from the open session, Google Drive, or an upload)

In [ ]:
def have_weights(d):
    return all(os.path.exists(os.path.join(d, n, "best.weights.h5"))
               for n in ["MobileNetV2", "EfficientNetB0", "Hybrid_EffNetB0_Transformer"])

if not have_weights(RESULTS_DIR):
    zips = glob.glob("/content/drive/MyDrive/**/unified_results*.zip", recursive=True) if os.path.isdir("/content/drive") else []
    if not zips:
        from google.colab import files
        print("Please choose unified_results.zip from your computer:")
        up = files.upload(); zips = [k for k in up if k.endswith(".zip")]
    os.makedirs(RESULTS_DIR, exist_ok=True)
    zipfile.ZipFile(zips[0]).extractall(RESULTS_DIR)
    # the zip may contain the files at its top level or inside a folder
    if not have_weights(RESULTS_DIR):
        inner = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(RESULTS_DIR + "/**/best.weights.h5", recursive=True)]
        RESULTS_DIR = inner[0]
assert have_weights(RESULTS_DIR), "Trained model weights not found"
print("Using trained models from:", RESULTS_DIR)

### Download and extract LungHist700

In [ ]:
if not glob.glob(DATA_ROOT + "/**/*.jpg", recursive=True):
    os.makedirs(DATA_ROOT, exist_ok=True)
    rar = "/content/LungHist700.rar"
    if not os.path.exists(rar):
        subprocess.run(["wget", "-q", "-O", rar, LH_URL], check=True)
    if subprocess.run(["which", "unrar"], capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
    r = subprocess.run(["unrar", "x", "-o+", "-inul", rar, DATA_ROOT + "/"])
    if r.returncode != 0:   # fallback extractor
        subprocess.run(["7z", "x", "-y", f"-o{DATA_ROOT}", rar], check=True, stdout=subprocess.DEVNULL)

# Build the image table. File names look like  aca_bd_20x_12.jpg / nor_40x_3.jpg / scc_pd_20x_7.jpg
rows = []
for f in sorted(glob.glob(DATA_ROOT + "/**/*.jp*g", recursive=True)):
    name = os.path.basename(f).lower()
    m = re.match(r"^(aca|nor|scc)(?:_(bd|md|pd))?_(20x|40x)_(\d+)\.jpe?g$", name)
    if not m:
        continue
    sup, sub, res, iid = m.groups()
    rows.append({"path": f, "superclass": sup, "subclass": sub or "", "resolution": res, "image_id": int(iid),
                 "label": {"aca": 0, "nor": 1, "scc": 2}[sup]})
ext = pd.DataFrame(rows)
if len(ext) == 0:
    print("Example files found:", glob.glob(DATA_ROOT + "/**/*.*", recursive=True)[:10])
    raise SystemExit("No LungHist700 images matched the expected names - send this message to Claude")
# patient IDs from the accompanying csv (used for patient-level confidence intervals)
csvs = [c for c in glob.glob(DATA_ROOT + "/**/*.csv", recursive=True)]
ext["patient_id"] = np.nan
for c in csvs:
    meta = pd.read_csv(c)
    if {"superclass", "resolution", "image_id", "patient_id"} <= set(meta.columns):
        meta = meta.copy()
        if "subclass" not in meta: meta["subclass"] = ""
        meta["subclass"] = meta["subclass"].fillna("")
        meta["image_id"] = meta["image_id"].astype(int)
        meta["superclass"] = meta["superclass"].astype(str).str.lower()
        meta["subclass"] = meta["subclass"].astype(str).str.lower().replace("nan", "")
        meta["resolution"] = meta["resolution"].astype(str).str.lower()
        ext = ext.drop(columns="patient_id").merge(meta[["superclass", "subclass", "resolution", "image_id", "patient_id"]],
                                                   on=["superclass", "subclass", "resolution", "image_id"], how="left")
        break
print(f"LungHist700 images: {len(ext)} | patients known for {ext.patient_id.notna().sum()} images")
print(pd.crosstab(ext["superclass"] + "_" + ext["subclass"].replace("", "-"), ext["resolution"], margins=True))
ext.to_csv(os.path.join(OUT_DIR, "lunghist700_index.csv"), index=False)

### Colour reference for Reinhard normalisation (mean/std in LAB space of LC25000 lung training-style images)

In [ ]:
def lab_stats(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    return lab.reshape(-1, 3).mean(0), lab.reshape(-1, 3).std(0)

def find_lc_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES):
            return d
    return None

LC_DIR = os.environ.get("EXT_LC") if TEST_MODE else None
if LC_DIR is None:
    LC_DIR = find_lc_dir("/root/.cache/kagglehub") if os.path.isdir("/root/.cache/kagglehub") else None
    if LC_DIR is None:
        import kagglehub
        LC_DIR = find_lc_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
rng = np.random.default_rng(SEED)
means, stds = [], []
for c in CLASSES:
    fs = sorted(glob.glob(os.path.join(LC_DIR, c, "*")))
    for f in rng.choice(fs, min(REINHARD_REF_PER_CLASS, len(fs)), replace=False):
        mu, sd = lab_stats(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); means.append(mu); stds.append(sd)
REF_MU, REF_SD = np.mean(means, 0), np.mean(stds, 0)
print("LC25000 LAB reference mean/std:", REF_MU.round(1), REF_SD.round(1))

def reinhard(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    mu, sd = lab.reshape(-1, 3).mean(0), lab.reshape(-1, 3).std(0) + 1e-6
    lab = (lab - mu) / sd * REF_SD + REF_MU
    return cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)

### Rebuild the three architectures exactly as trained, and load the trained weights

In [ ]:
def head(x):
    x = layers.Dropout(0.3, name="dropout")(x)
    return layers.Activation("softmax", name="pred")(layers.Dense(3, name="logits")(x))

def build_mobilenet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1 / 127.5, offset=-1, name="mnv2_preprocess")(inp)
    base = keras.applications.MobileNetV2(include_top=False, weights=None, input_tensor=x)
    return keras.Model(inp, head(layers.GlobalAveragePooling2D(name="gap")(base.output)), name="MobileNetV2")

def build_efficientnet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=None, input_tensor=inp)
    return keras.Model(inp, head(layers.GlobalAveragePooling2D(name="gap")(base.output)), name="EfficientNetB0")

def build_hybrid():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=None, input_tensor=inp)
    f = base.output; h, w, c = f.shape[1], f.shape[2], f.shape[3]
    tok = layers.Reshape((h * w, c), name="tokens")(f)
    t = layers.LayerNormalization(epsilon=1e-6, name="ln1")(tok)
    a, _ = layers.MultiHeadAttention(num_heads=2, key_dim=16, name="mhsa")(t, t, return_attention_scores=True)
    x = layers.Add(name="res1")([tok, a])
    t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(x)
    t = layers.Dense(c, name="mlp2")(layers.Dense(128, activation="gelu", name="mlp1")(t))
    x = layers.Add(name="res2")([x, t])
    return keras.Model(inp, head(layers.GlobalAveragePooling1D(name="gap")(x)), name="Hybrid_EffNetB0_Transformer")

MODELS = {}
for name, b in [("MobileNetV2", build_mobilenet), ("EfficientNetB0", build_efficientnet),
                ("Hybrid_EffNetB0_Transformer", build_hybrid)]:
    m = b(); m.load_weights(os.path.join(RESULTS_DIR, name, "best.weights.h5")); MODELS[name] = m
    print(name, "loaded,", m.count_params(), "parameters")
names = list(MODELS)

### Inference under four conditions: {tiled, whole image} x {no normalisation, Reinhard}

In [ ]:
def make_inputs(img_rgb, mode):
    H, W = img_rgb.shape[:2]
    if mode == "whole":   # centre square crop of the full field, resized
        s = min(H, W); y0, x0 = (H - s) // 2, (W - s) // 2
        crops = [img_rgb[y0:y0 + s, x0:x0 + s]]
    else:                 # non-overlapping TILE x TILE tiles covering the image
        crops = [img_rgb[y:y + TILE, x:x + TILE] for y in range(0, H - TILE + 1, TILE) for x in range(0, W - TILE + 1, TILE)]
    return np.stack([cv2.resize(c, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA) for c in crops]).astype(np.float32)

predict = {n: tf.function(lambda x, m=m: m(x, training=False), reduce_retracing=True) for n, m in MODELS.items()}
CONDS = [("tiled", "none"), ("tiled", "reinhard"), ("whole", "none"), ("whole", "reinhard")]
probs = {(n, mode, norm): np.zeros((len(ext), 3), np.float32) for n in names for mode, norm in CONDS}
for i, f in enumerate(ext["path"]):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)
    for norm in ["none", "reinhard"]:
        im = img if norm == "none" else reinhard(img)
        for mode in ["tiled", "whole"]:
            x = make_inputs(im, mode)
            for n in names:
                probs[(n, mode, norm)][i] = predict[n](tf.constant(x)).numpy().mean(0)
    if (i + 1) % 100 == 0: print(f"  {i + 1}/{len(ext)} images")

### Metrics, confidence intervals and McNemar tests

In [ ]:
y = ext["label"].values
groups = ext["patient_id"].values if ext["patient_id"].notna().all() else None

def ci(correct, groups=None):
    rng = np.random.default_rng(SEED); accs = []
    if groups is None:
        for _ in range(N_BOOT): accs.append(correct[rng.integers(0, len(correct), len(correct))].mean())
    else:   # patient-level (cluster) bootstrap
        pats = np.unique(groups); idx = {p: np.where(groups == p)[0] for p in pats}
        for _ in range(N_BOOT):
            s = np.concatenate([idx[p] for p in rng.choice(pats, len(pats), replace=True)]); accs.append(correct[s].mean())
    return [float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))]

results = {}
for mode, norm in CONDS:
    for subset in ["all", "20x", "40x"]:
        mask = np.ones(len(ext), bool) if subset == "all" else (ext["resolution"].values == subset)
        key = f"{mode}|{norm}|{subset}"; results[key] = {"n_images": int(mask.sum()), "models": {}, "mcnemar": {}}
        preds = {}
        for n in names:
            p = probs[(n, mode, norm)][mask]; yp = p.argmax(1); yt = y[mask]; preds[n] = yp
            correct = (yp == yt).astype(float)
            rep = classification_report(yt, yp, labels=[0, 1, 2], target_names=CLASSES, output_dict=True, zero_division=0)
            results[key]["models"][n] = {
                "accuracy": float(correct.mean()), "accuracy_95ci": ci(correct, None if groups is None else groups[mask]),
                "ci_type": "patient-level bootstrap" if groups is not None else "image-level bootstrap",
                "macro_f1": float(f1_score(yt, yp, average="macro", labels=[0, 1, 2], zero_division=0)),
                "recall": {c: round(rep[c]["recall"], 4) for c in CLASSES},
                "precision": {c: round(rep[c]["precision"], 4) for c in CLASSES},
                "confusion_matrix": confusion_matrix(yt, yp, labels=[0, 1, 2]).tolist()}
        for i in range(3):
            for j in range(i + 1, 3):
                a, b = preds[names[i]] == y[mask], preds[names[j]] == y[mask]
                t = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
                results[key]["mcnemar"][f"{names[i]} vs {names[j]}"] = {"table": t, "p_value": float(mcnemar(t, exact=True).pvalue)}

# per-grade accuracy for the primary analysis (are some differentiation grades harder?)
grade = (ext["superclass"] + "_" + ext["subclass"].replace("", "-")).values
per_grade = {n: {g: float((probs[(n, "tiled", "none")].argmax(1)[grade == g] == y[grade == g]).mean())
                 for g in sorted(set(grade))} for n in names}
pd.DataFrame({n: probs[(n, "tiled", "none")].argmax(1) for n in names}).assign(
    path=ext["path"].values, y_true=y, resolution=ext["resolution"].values, grade=grade).to_csv(
    os.path.join(OUT_DIR, "predictions_primary.csv"), index=False)

### Tables and figures (shown here and saved)

In [ ]:
from IPython.display import display, Markdown
short = {"MobileNetV2": "MobileNetV2", "EfficientNetB0": "EfficientNetB0", "Hybrid_EffNetB0_Transformer": "Hybrid"}
tab = pd.DataFrame([{"Inference": mode, "Colour norm.": norm, "Subset": sub,
                     **{short[n]: f'{100*results[f"{mode}|{norm}|{sub}"]["models"][n]["accuracy"]:.1f}' for n in names}}
                    for mode, norm in CONDS for sub in ["all", "20x", "40x"]])
display(Markdown("## External accuracy (%) on LungHist700 – all conditions")); display(tab)
prim = results["tiled|none|all"]
display(Markdown("## Primary analysis (tiled, no colour normalisation, all images)"))
display(pd.DataFrame({short[n]: {"Accuracy (%)": round(100*m["accuracy"], 2),
                                 "95% CI (%)": f'{100*m["accuracy_95ci"][0]:.1f}–{100*m["accuracy_95ci"][1]:.1f}',
                                 "Macro-F1": round(m["macro_f1"], 4),
                                 **{f"Recall {c}": m["recall"][c] for c in CLASSES}} for n, m in prim["models"].items()}).T)
display(pd.DataFrame([{"Comparison": k, "Only first correct": v["table"][0][1], "Only second correct": v["table"][1][0],
                       "p-value": round(v["p_value"], 4)} for k, v in prim["mcnemar"].items()]).set_index("Comparison"))
display(Markdown("## Accuracy by class and differentiation grade (primary analysis)")); display(pd.DataFrame(per_grade).round(3))

plt.rcParams.update({"font.family": "serif", "font.size": 10})
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, n in zip(axes, names):
    cm = np.array(prim["models"][n]["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for (r, c), v in np.ndenumerate(cm):
        ax.text(c, r, v, ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f'{short[n]} ({100*prim["models"][n]["accuracy"]:.1f}%)')
fig.suptitle(f"External test on LungHist700 (n = {prim['n_images']}, tiled inference)"); fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig_external_confusion.png"), dpi=300, bbox_inches="tight"); plt.show() if not TEST_MODE else None; plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 3.8)); w = 0.25; labels = [f"{m}\n{n_}" for m, n_ in CONDS]
for k, (n, col) in enumerate(zip(names, ["#9e9e9e", "#5b7fa6", "#1f4e79"])):
    vals = [100 * results[f"{m}|{n_}|all"]["models"][n]["accuracy"] for m, n_ in CONDS]
    errs = np.array([[100 * (results[f"{m}|{n_}|all"]["models"][n]["accuracy"] - results[f"{m}|{n_}|all"]["models"][n]["accuracy_95ci"][0]),
                      100 * (results[f"{m}|{n_}|all"]["models"][n]["accuracy_95ci"][1] - results[f"{m}|{n_}|all"]["models"][n]["accuracy"])]
                     for m, n_ in CONDS]).T
    ax.bar(np.arange(4) + (k - 1) * w, vals, w * 0.92, yerr=errs, capsize=3, color=col, label=short[n])
ax.axhline(100 / 3, ls="--", color="#777", lw=1); ax.text(3.45, 100 / 3 + 1, "chance", fontsize=8, color="#555")
ax.set_xticks(range(4), [l.replace("tiled", "Tiled").replace("whole", "Whole image").replace("none", "no normalisation").replace("reinhard", "Reinhard") for l in labels])
ax.set_ylabel("External accuracy (%)"); ax.set_ylim(0, 100); ax.legend(frameon=False, ncol=3, loc="upper left")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fig_external_conditions.png"), dpi=300, bbox_inches="tight")
plt.show() if not TEST_MODE else None; plt.close(fig)

### Save and print the summary for Claude

In [ ]:
full = {"dataset": {"n_images": int(len(ext)), "by_class": ext["superclass"].value_counts().to_dict(),
                    "by_resolution": ext["resolution"].value_counts().to_dict(),
                    "n_patients": int(ext["patient_id"].nunique()) if ext["patient_id"].notna().any() else None},
        "settings": {"tile": TILE, "img_size": IMG_SIZE, "reinhard_ref_images": 3 * REINHARD_REF_PER_CLASS,
                     "ref_lab_mean": REF_MU.round(2).tolist(), "ref_lab_std": REF_SD.round(2).tolist()},
        "results": results, "per_grade_accuracy_primary": per_grade}
json.dump(full, open(os.path.join(OUT_DIR, "external_summary.json"), "w"), indent=1)
shutil.make_archive(OUT_DIR.rstrip("/"), "zip", OUT_DIR)
compact = {"dataset": full["dataset"],
           "accuracy_table": {k: {short[n]: [round(100*m["accuracy"], 2), [round(100*v, 1) for v in m["accuracy_95ci"]]]
                                  for n, m in r["models"].items()} for k, r in results.items()},
           "primary": results["tiled|none|all"], "per_grade_accuracy_primary": per_grade}
print("\n" + "=" * 70 + "\nSUMMARY — copy everything below and send it to Claude\n" + "=" * 70)
print(json.dumps(compact, indent=1))
if not TEST_MODE:
    try:
        from google.colab import files; files.download(OUT_DIR.rstrip("/") + ".zip")
    except Exception:
        print("Download external_results.zip from the Files panel on the left.")